# Low-Rank Analysis

Understand the **effective dimensionality** of model weight matrices.
How much of each weight's capacity is actually used? Can we approximate
weights with lower-rank versions without losing important behavior?

This notebook covers the `irtk.low_rank` module.

## Why This Matters

Low-rank analysis reveals the effective dimensionality of weight matrices and attention heads. Many components have surprisingly low rank, meaning they compute in a lower-dimensional subspace than their full capacity would allow. This constrains what computations are possible.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import low_rank

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Weight SVD

Decompose any weight matrix to understand its spectral structure.

In [ ]:
svd = low_rank.weight_svd(model, "blocks.0.mlp.W_in")
print(f"Shape: {svd['shape']}")
print(f"Numerical rank: {svd['rank']}")
print(f"Top 10 singular values: {svd['S'][:10].round(4)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(svd['S'], 'b-')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Singular value')
axes[0].set_title('Singular Value Spectrum (MLP W_in, Layer 0)')

axes[1].semilogy(svd['S'], 'b-')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('Singular value (log scale)')
axes[1].set_title('Log Spectrum')

plt.tight_layout()
plt.show()

## 2. Effective Rank

How many singular values are needed to capture most of the energy?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for layer in range(model.cfg.n_layers):
    er = low_rank.effective_rank(model, f"blocks.{layer}.mlp.W_in")
    axes[0].plot(er['cumulative_energy'], label=f'L{layer}')

axes[0].axhline(0.99, color='red', linestyle='--', label='99%')
axes[0].axhline(0.95, color='orange', linestyle='--', label='95%')
axes[0].set_xlabel('Number of singular values')
axes[0].set_ylabel('Cumulative energy fraction')
axes[0].set_title('MLP W_in Cumulative Energy by Layer')
axes[0].legend(fontsize=7, ncol=3)

# Effective ranks across layers
ranks_99 = []
ranks_95 = []
for layer in range(model.cfg.n_layers):
    r99 = low_rank.effective_rank(model, f"blocks.{layer}.mlp.W_in", threshold=0.99)
    r95 = low_rank.effective_rank(model, f"blocks.{layer}.mlp.W_in", threshold=0.95)
    ranks_99.append(r99['effective_rank'])
    ranks_95.append(r95['effective_rank'])

x = range(model.cfg.n_layers)
axes[1].bar(x, ranks_99, alpha=0.7, label='99% energy', color='steelblue')
axes[1].bar(x, ranks_95, alpha=0.7, label='95% energy', color='coral')
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Effective rank')
axes[1].set_title('MLP W_in Effective Rank by Layer')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Low-Rank Approximation

How much error does truncating to rank-k introduce?

In [ ]:
ranks = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
errors = []
energies = []
for r in ranks:
    result = low_rank.low_rank_approximation(model, "blocks.6.mlp.W_in", rank=r)
    errors.append(result['relative_error'])
    energies.append(result['energy_captured'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogx(ranks, errors, 'bo-')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Relative reconstruction error')
axes[0].set_title('Reconstruction Error vs Rank (Layer 6 MLP W_in)')

axes[1].semilogx(ranks, energies, 'ro-')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Energy captured')
axes[1].set_title('Energy Captured vs Rank')
axes[1].axhline(0.99, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Spectrum Similarity

Do different layers use their weight capacity in similar ways?

In [ ]:
n = model.cfg.n_layers
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        r = low_rank.weight_spectrum_similarity(
            model, f"blocks.{i}.mlp.W_in", f"blocks.{j}.mlp.W_in"
        )
        sim_matrix[i, j] = r['spectral_similarity']
        sim_matrix[j, i] = r['spectral_similarity']

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xlabel('Layer')
ax.set_ylabel('Layer')
ax.set_title('MLP W_in Spectral Similarity Across Layers')
plt.colorbar(im, ax=ax, label='Cosine similarity of SV spectra')
plt.tight_layout()
plt.show()

## 5. Apply Low-Rank Weights

Replace a weight with its rank-k approximation and measure the impact.

In [ ]:
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)

clean_logits = model(tokens)
clean_probs = jax.nn.softmax(clean_logits[-1])
paris_id = tokenizer.encode(" Paris")[0]

print(f"Clean P(' Paris'): {float(clean_probs[paris_id]):.4f}\n")

for rank in [4, 16, 64, 256, 768]:
    modified = low_rank.apply_low_rank_weights(model, "blocks.6.mlp.W_in", rank=rank)
    mod_logits = modified(tokens)
    mod_probs = jax.nn.softmax(mod_logits[-1])
    kl = float(jnp.sum(clean_probs * (jnp.log(clean_probs + 1e-10) - jnp.log(mod_probs + 1e-10))))
    print(f"  Rank {rank:4d}: P(' Paris')={float(mod_probs[paris_id]):.4f}  KL={kl:.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `weight_svd()` | SVD of any named weight matrix |
| `effective_rank()` | Min SVs to capture threshold energy |
| `low_rank_approximation()` | Rank-k truncated SVD reconstruction |
| `weight_spectrum_similarity()` | Compare spectral structure across weights |
| `apply_low_rank_weights()` | Replace weight with rank-k approximation |